<a href="https://colab.research.google.com/github/uzair848/AI/blob/main/1BM24CS144_week2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import collections
import time
import math

def get_neighbors(state):
    """Generates all valid moves (Up, Down, Left, Right) from the current state."""
    neighbors = []
    zero_idx = state.index(0)  # Find the empty space
    row, col = divmod(zero_idx, 3)  # Convert 1D index to 2D coordinates

    # Define possible directional shifts: (row_change, col_change, move_name)
    moves = [(-1, 0, 'Up'), (0, -1, 'Left'), (0, 1, 'Right'),(1, 0, 'Down'),]

    for dr, dc, move_name in moves:
        new_row, new_col = row + dr, col + dc

        # Check boundary rules for a 3x3 grid
        if 0 <= new_row < 3 and 0 <= new_col < 3:
            new_idx = new_row * 3 + new_col
            neighbor_lst = list(state)
            neighbor_lst[zero_idx], neighbor_lst[new_idx] = neighbor_lst[new_idx], neighbor_lst[zero_idx]
            neighbors.append((tuple(neighbor_lst), move_name))

    return neighbors


def solve_8_puzzle_dfs(start_state, goal_state, max_depth=30):
    """Solves the 8-puzzle using Depth-First Search with a depth limit.
       Returns path, explored_states_count, time_comp, space_comp.
    """
    start_time = time.time()

    stack = collections.deque([(start_state, [], 0)]) # (state, path_to_state, depth)
    visited = set()

    explored_states_count = 0
    max_stack_size = 0
    max_visited_size = 0

    while stack:
        max_stack_size = max(max_stack_size, len(stack))

        current_state, path, depth = stack.pop()
        explored_states_count += 1

        if current_state == goal_state:
            end_time = time.time()
            time_comp = end_time - start_time
            # For DFS, space complexity is primarily the max size of the stack and visited set
            space_comp = max_stack_size + max_visited_size
            return path, explored_states_count, time_comp, space_comp

        if current_state in visited or depth >= max_depth:
            continue

        visited.add(current_state)
        max_visited_size = max(max_visited_size, len(visited))

        # Explore neighbors in reverse order to simulate typical DFS behavior (e.g., Up, Left first)
        # Or, just iterate if order doesn't matter for specific problem type.
        for neighbor_state, move in reversed(get_neighbors(current_state)):
            if neighbor_state not in visited:
                stack.append((neighbor_state, path + [move], depth + 1))

    end_time = time.time()
    time_comp = end_time - start_time
    space_comp = max_stack_size + max_visited_size
    return None, explored_states_count, time_comp, space_comp


def print_board(state):
    """Formats and prints the 1D puzzle representation into a 3x3 layout."""
    for i in range(0, 9, 3):
        print(tuple(state[i:i+3])) # Print as tuple rows for consistency with input
    print("-" * 15)


def get_user_input(prompt_name):
    """Prompts user for input as a 3x3 matrix, validates it, and converts it to a 9-tuple."""
    while True:
        print(f"\nEnter the {prompt_name} ( use 0 for blank tile,):")
        matrix_rows = []
        all_numbers = []
        try:
            for i in range(3):
                row_str = input(f"Row {i+1} ").strip()
                row = tuple(map(int, row_str.split()))
                if len(row) != 3:
                    raise ValueError(f"Expected 3 numbers for Row {i+1}, but got {len(row)}.")
                matrix_rows.append(row)
                all_numbers.extend(row)

            # Validate numbers: must be unique and between 0 and 8
            if len(set(all_numbers)) != 9 or not all(0 <= n <= 8 for n in all_numbers):
                raise ValueError("Invalid input: The 3x3 matrix must contain unique numbers from 0 to 8 (inclusive).")

            return tuple(all_numbers) # Convert to 1D tuple for the solver
        except ValueError as e:
            print(f"Error: {e}. Please try again.")


# --- Execution ---
if __name__ == "__main__":

    # Get custom states from the user
    initial_state = get_user_input("INITIAL STATE")
    target_state = get_user_input("GOAL STATE")

    print("\nInitial Configuration:")
    print_board(initial_state)

    print("Goal Configuration:")
    print_board(target_state)

    depth_limit = 25 # Note: High max_depth values might cause pure DFS to consume too much memory or hang
    print(f"Searching for a solution using DFS (Depth limit: {depth_limit})...\n")

    solution_path, explored_states_count, time_comp, space_comp = solve_8_puzzle_dfs(initial_state, target_state, max_depth=depth_limit)

    if solution_path is not None:
        solution_depth = len(solution_path)
        print(f"Success! Goal reached in {solution_depth} steps.")
        print("Sequence of moves:", " -> ".join(solution_path))
        print("\nPath to Goal (showing states after each move):")

        # Reconstruct and print states in path (optional, but good for visualization)
        current_temp_state = initial_state
        print_board(current_temp_state)
        for move in solution_path:
            for neighbor_state, neighbor_move in get_neighbors(current_temp_state):
                if neighbor_move == move:
                    current_temp_state = neighbor_state
                    print(f"Move: {move}")
                    print_board(current_temp_state)
                    break

        print(f"Execution Time: {time_comp:.4f} seconds")

        # Theoretical Complexities (O(b^d))
        branching_factor = 3 # User requested branching factor of 3
        theoretical_time_nodes = branching_factor ** solution_depth
        theoretical_space_nodes = theoretical_time_nodes # In worst-case, DFS might keep all explored nodes in memory for visited set/stack

        print(f"\n---  Complexities (O(b^d)) ---")
        print(f"For Solution Depth (d) = {solution_depth}, Branching Factor (b) = {branching_factor}")
        print(f"Time Complexity: {theoretical_time_nodes}")
        print(f"Space Complexity: {theoretical_space_nodes}")

    else:
        print(f"\nCould not find a solution within the depth limit of {depth_limit} steps.")
        print("Tip: Because DFS explores paths blindly, it often fails on complex configurations or requires a much larger depth limit.")
        print(f"Total Explored States: {explored_states_count}")
        print(f"Execution Time: {time_comp:.4f} seconds")


Enter the INITIAL STATE ( use 0 for blank tile,):
Row 1 2 8 3
Row 2 1 6 4
Row 3 7 0 5

Enter the GOAL STATE ( use 0 for blank tile,):
Row 1 1 2 3
Row 2 8 0 4
Row 3 7 6 5

Initial Configuration:
(2, 8, 3)
(1, 6, 4)
(7, 0, 5)
---------------
Goal Configuration:
(1, 2, 3)
(8, 0, 4)
(7, 6, 5)
---------------
Searching for a solution using DFS (Depth limit: 25)...

Success! Goal reached in 5 steps.
Sequence of moves: Up -> Up -> Left -> Down -> Right

Path to Goal (showing states after each move):
(2, 8, 3)
(1, 6, 4)
(7, 0, 5)
---------------
Move: Up
(2, 8, 3)
(1, 0, 4)
(7, 6, 5)
---------------
Move: Up
(2, 0, 3)
(1, 8, 4)
(7, 6, 5)
---------------
Move: Left
(0, 2, 3)
(1, 8, 4)
(7, 6, 5)
---------------
Move: Down
(1, 2, 3)
(0, 8, 4)
(7, 6, 5)
---------------
Move: Right
(1, 2, 3)
(8, 0, 4)
(7, 6, 5)
---------------
Execution Time: 0.0001 seconds

---  Complexities (O(b^d)) ---
For Solution Depth (d) = 5, Branching Factor (b) = 3
Time Complexity: 243
Space Complexity: 243
